In [ ]:
from unstructured.partition.pdf import partition_pdf

from unstructured.documents.elements import (
    Element,
    Text,
    Image,
    FigureCaption
)

print("All imports successful!")

In [ ]:
base_dir ="files"
pdf_file_name ="2312.10997v5.pdf"
pdf_file_path = f"{base_dir}/{pdf_file_name}"

In [ ]:
import os
import shutil

poppler_path = r"C:\Users\psran\Downloads\poppler-26.09.0\Library\bin"

os.environ["PATH"] = poppler_path + os.pathsep + os.environ["PATH"]

print("pdfinfo:", shutil.which("pdfinfo"))
print("pdftoppm:", shutil.which("pdftoppm"))

In [ ]:
import os
import shutil

tesseract_path = r"C:\Program Files\Tesseract-OCR"

os.environ["PATH"] = tesseract_path + os.pathsep + os.environ["PATH"]

print("Tesseract path:", shutil.which("tesseract"))

In [ ]:
raw_chunks = partition_pdf(
    filename=pdf_file_path,
    strategy="hi_res",
    infer_table_structure=True,
    extract_image_block_types=["Image", "Table"],
    extract_image_block_to_payload=True,
)

print(f"Total elements extracted: {len(raw_chunks)}")

raw_chunks

In [ ]:
# Print the indices of chunks that are images
all_image_indices = []
for idx, chunk in enumerate(raw_chunks):
    if isinstance(chunk, Image):
        all_image_indices.append(idx)
        print(idx)

In [ ]:
first_image_chunk_idx = all_image_indices[0]
raw_chunks[first_image_chunk_idx].to_dict()

In [ ]:
raw_chunks[32].to_dict()

In [ ]:
all_images = []

for idx, chunk in enumerate(raw_chunks):
    if isinstance(chunk, Image):
        if idx + 1 <len(raw_chunks) and isinstance(raw_chunks[idx + 1 ], FigureCaption):
            caption = raw_chunks[idx + 1 ].text
        else:
            caption = None

        all_images.append({
            "index": idx,
            "caption": caption if caption else "No caption",
            "image_text": chunk.text,
            "base64_image": chunk.metadata.image_base64,
        })

    

In [ ]:
all_images[0]

In [ ]:
import base64
from IPython.display import display, Image as Ipimage
def display_image(image_based64):
    image_data = base64.b64decode(image_based64)
    display(Ipimage(data=image_data, format="png"))

In [ ]:
for image_data in all_images:
    print(f"Index: {image_data['index']}, Caption: {image_data['caption']}")
    display_image(image_data["base64_image"])


In [ ]:
from dotenv import  load_dotenv
load_dotenv()
import os

api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY is not set in the enviroment variable")

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--upgrade",
    "google-generativeai"
])

In [ ]:

import base64
from google import genai

client = genai.Client(api_key=api_key)

def genrate_image_description(image_data):
    image_binary = base64.b64decode(image_data["base64_image"])

    prompt = (
        "Describe this image in detail.\n"
        f"Caption: {image_data.get('caption', '')}\n"
        f"Image text: {image_data.get('image_text', '')}"
    )

    response = client.models.generate_content(
        model="gemini-3.8-flash",
        contents=[
            prompt,
            {
                "inline_data": {
                    "mime_type": "image/jpeg",
                    "data": image_binary
                }
            }
        ]
    )

    return response.text


In [ ]:
print(genrate_image_description(all_images[0]))

Table Parsing

In [ ]:
from unstructured.documents.elements import  Table

table_data = []
for idx, element in enumerate(raw_chunks):
    if isinstance(element, Table):
        table_data.append({
            "table_as_html": element.metadata.text_as_html,
            
        })
        # print(element.to_dict()['metadata'].keys())
        # break


In [ ]:
table_data[0]

In [ ]:

from IPython.display import display, HTML

def display_table_as_html(table_html):
    display(HTML(table_html))

display_table_as_html(table_data[1]["table_as_html"])




In [ ]:
table_data[1]

In [ ]:

from google import genai

client = genai.Client(api_key=api_key)

def genrate_image_description(table_html):
    prompt = (
        "Analyze the following HTML table and provide a detailed description "
        "of its contents, including its structure, key data points, "
        "notable trends, and insights.\n\n"
        f"Table in HTML format:\n{table_html}\n\n"
        "Analyze the table directly without additional introductory text."
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text



In [ ]:
print(genrate_image_description(table_data[1]["table_as_html"]))

In [ ]:
import requests
def genrate_table_summary(table_html):
    url = "http://localhost:11434/api/generate"

    headers = {
        "Content-Type": "application/json",
    }

    data= {
        "model": "deepseek-r1:1.5b",
        "prompt":(
            "Analyze the folowing table and provide a detailed summary of it's content. "
            "including the structure, key data point, and any notable treands or insight. "
            f"Here is the table in Html format: {table_html}"
            "Directly analyze the table and provide a detailed summary without any additional"
        ),
        "max_token": 1000,
    }

    response = requests.post(url, headers=headers, json=data)
    response.raise_for_status()
    return response.json()["response"]



In [ ]:
genrate_table_summary(table_data[1]["table_as_html"])

In [ ]:
text_chunk = partition_pdf(
        filename=pdf_file_path,
        strategy="hi_res",
        chunking_strategy="by_title",
        max_characters = 2000,
        combine_text_under_n_chars = 500,
        nw_after_n_char = 1500
    )
